# Environment test - Drake + Meshcat + airo-mono setup

Confirms the MVP 1 & 2 environment (`../src/environment.yaml` on Linux, `../src/environment-macos.yaml` on macOS) is working:

1. **Import check** - core packages load.
2. **MVP item 1** - load UR3e + Robotiq 2F-85 URDFs into Drake, place a lego brick, and run a kinematic grab simulation, visualized in Meshcat.
3. **MVP item 2** - rotate the wrist through a few configurations, publish to Meshcat, and render + save camera frames (a tiny pretrain-dataset stub).
4. **MVP item 3** - attempt to reach the real UR3 (and its RealSense camera). This section is expected to fail unless you're on the robot's network; see the markdown notes for platform requirements (camera capture needs Linux).

**Before running:** create and activate the conda env, then select the `irm` kernel in Jupyter.
```
conda env create -f ../src/environment.yaml        # Linux
conda env create -f ../src/environment-macos.yaml  # macOS
conda activate irm
python -m ipykernel install --user --name irm --display-name "Python (irm)"
```

In [1]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

Python: 3.11.15 | packaged by conda-forge | (main, Jun 11 2026, 03:29:05) [Clang 19.1.7 ]
Platform: macOS-26.5.2-arm64-arm-64bit


## 1. Import check

In [2]:
import importlib

packages = [
    "numpy", "scipy", "matplotlib", "loguru", "pydrake",
    "airo_models", "airo_drake", "airo_typing", "airo_spatial_algebra", "PIL",
]
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"OK   {pkg}")
    except ImportError as e:
        print(f"FAIL {pkg}: {e}")

OK   numpy
OK   scipy
OK   matplotlib
OK   loguru
OK   pydrake
OK   airo_models
OK   airo_drake
OK   airo_typing
OK   airo_spatial_algebra
OK   PIL


In [5]:
# Make the sibling src/ modules (scene.py, grab_demo.py, wrist_render.py) importable
# regardless of whether this notebook is launched from src/ or the repo root.
import sys
from pathlib import Path

SRC_DIR = Path.cwd() if (Path.cwd() / "scene.py").exists() else Path.cwd() / "src"
assert (SRC_DIR / "scene.py").exists(), f"Could not locate src/ modules from {Path.cwd()}"
sys.path.insert(0, str(SRC_DIR))
print("Using src dir:", SRC_DIR)

Using src dir: /Users/chaseungjun/Library/Mobile Documents/com~apple~CloudDocs/CSE/UGhent/int2026/src


## 2. MVP item 1 - Drake gripper + grab simulation

Loads the UR3e arm and Robotiq 2F-85 gripper URDFs (via `airo_models`) into Drake, welds the gripper to the arm's `tool0`, places a lego brick (from `../lego_3d/urdf/`) at the gripper's TCP, and kinematically closes the fingers around it. Open the printed Meshcat URL in a browser to watch it live.

In [6]:
from pydrake.geometry import Meshcat
from grab_demo import run_grab_demo

meshcat_grab = Meshcat()
print(f"Open in your browser: {meshcat_grab.web_url()}")

INFO:drake:Meshcat listening for connections at http://localhost:7001


Open in your browser: http://localhost:7001


In [7]:
run_grab_demo(meshcat_grab, publish_delay=0.3)
print("Grab simulation complete: the Robotiq gripper closed around the lego brick.")

Grab simulation complete: the Robotiq gripper closed around the lego brick.


## 3. MVP item 2 - rotate wrist, render, and save frames (pretrain stub)

Sweeps `wrist_3_joint` through 5 angles, publishing each pose to Meshcat and rendering an RGB frame from a fixed camera via Drake's VTK render engine. Frames are saved to `src/data/pretrain_frames/` as a tiny image dataset stub for later perception pretraining.

In [ ]:
from wrist_render import run_wrist_render, FRAMES_DIR

meshcat_wrist = Meshcat()
print(f"Open in your browser: {meshcat_wrist.web_url()}")
frames = run_wrist_render(meshcat_wrist, publish_delay=0.3)
print(f"Rendered and saved {len(frames)} frames to {FRAMES_DIR}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(frames), figsize=(4 * len(frames), 4))
for ax, frame in zip(axes, frames):
    ax.imshow(frame[..., :3])
    ax.axis("off")
plt.show()

## 4. MVP item 3 - real UR3 (and camera) connectivity

This section will only succeed on a machine that's on the same network as the robot controller (IP `10.42.0.162`, per the top-level `README.md`).

**Platform notes:**
- Robot arm control (`airo_robots`'s `URrtde`, wrapping the `ur_rtde` C++ library) builds and imports fine on **both Linux and macOS** (verified: `ur_rtde` compiles from source via pip on macOS). Whether it can actually *connect* only depends on network reachability, not OS.
- The RealSense camera (`airo_camera_toolkit`'s `Realsense`, needed for frame capture) depends on `pyrealsense2`, which Intel does **not** distribute for macOS at all (no wheels, no sdist). **Camera frame capture requires Linux** (or Windows) - use your Linux laptop for this part.

In [ ]:
import socket

import numpy as np

ROBOT_IP = "10.42.0.162"  # from README.md
RTDE_PORT = 30004  # UR RTDE interface port


def robot_reachable(ip, port, timeout=2.0):
    # URrtde(...) blocks for a long time (observed 2+ minutes) if the host is
    # unreachable, instead of failing fast - so probe the TCP port ourselves first.
    try:
        with socket.create_connection((ip, port), timeout=timeout):
            return True
    except OSError:
        return False


if not robot_reachable(ROBOT_IP, RTDE_PORT):
    print(f"Could not reach {ROBOT_IP}:{RTDE_PORT} within 2s - robot not on this network.")
    print("Expected unless this machine is on the lab/robot network. Test this on-site.")
else:
    try:
        from airo_robots.manipulators.hardware.ur_rtde import URrtde
        robot = URrtde(ROBOT_IP, URrtde.UR3E_CONFIG)
        q = robot.get_joint_configuration()
        print("Connected to UR3. Current joint configuration (rad):", np.round(q, 3))
    except ImportError as e:
        print(f"airo_robots / ur_rtde not installed in this environment: {e}")
    except Exception as e:
        print(f"Reached {ROBOT_IP}:{RTDE_PORT} but URrtde failed to connect: {e}")

In [ ]:
try:
    from airo_camera_toolkit.cameras.realsense.realsense import Realsense
    camera = Realsense()
    image = camera.get_rgb_image_as_int()
    print("Connected to RealSense camera. Frame shape:", image.shape)
except ImportError as e:
    print(f"airo_camera_toolkit / pyrealsense2 not available in this environment: {e}")
    print("pyrealsense2 has no macOS distribution - run this cell on Linux.")
except Exception as e:
    print(f"Camera library imported OK but could not open a device: {e}")